In [53]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [54]:
df = pd.read_csv('homework_7.1.csv')
df.drop(columns=['Unnamed: 0'], inplace=True)
df

,X,W,Z,Y
0,1.137055,1.221768,0.327829,1.944532
1,-0.112905,0.465835,0.599650,0.655514
2,2.077755,1.795414,-0.063393,5.934411
3,0.456373,-0.512159,1.177413,-0.188064
4,-1.012402,0.080002,-0.275697,-0.533775
...,...,...,...,...
9995,2.569934,1.233620,0.930467,6.188783
9996,0.190150,1.022164,-0.015151,0.697187
9997,-1.184465,-1.475929,-0.287056,-1.575303
9998,-0.121286,-0.914357,1.706237,-1.809819


### Problem 1

Q1. Suppose that a process can be modeled via linear regression: 

```
W = np.random.normal(0, 1, (1000,))
X = W + np.random.normal(0, 1, (1000,)) 
Z = np.random.normal(0, 1, (1000,)) 
Y = X + Z + W + np.random.normal(0, 1, (1000,))
```

Which is closest to the correlation of *X* with the error term in the equation for *Y*? 

Option A: -0.50

Option B: 0

Option C: 1.0

Option D: 0.75

Option E: 0.5


In [55]:
W = np.random.normal(0, 1, (1000,))
X = W + np.random.normal(0, 1, (1000,)) 
Z = np.random.normal(0, 1, (1000,)) 
Y = X + Z + W + np.random.normal(0, 1, (1000,))

In [56]:
err = Y - X
np.corrcoef(X, err)

array([[1.        , 0.37005423],
       [0.37005423, 1.        ]])

A1. Option E (0.5)

### Problem 2

Q2: If *Y* is written as depending on *X* and *Z* only, *W* is part of the error term. Which, then, is closest to the expected correlation of *X* with the error term in the equation for *Y*?

Option A: 0.75

Option B: 0.50

Option C: 1.0

Option D: -0.50

Option E: 0

In [57]:
err = Y - X - Z

# Calculate the correlation between X and the error
corr_matrix = np.corrcoef(X, err)
corr_matrix

array([[1.        , 0.46236415],
       [0.46236415, 1.        ]])

A2. Option B. (0.5)

### Problem 3

Q3. In the data frame for homework_7.1.csv, control for W by regressing *Y* on X
 and ﻿Z﻿ at the following constant values of ﻿W﻿: -1, 0, and 1. (You cannot literally use a constant value of ﻿W﻿, of course, or you will have only one data point! How will you manage this?) The question is: Is the coefficient of ﻿X﻿

Option A. decreasing

Option B. staying about the same (say, within 0.2 or so) as﻿W﻿ increases? 

Option C. increasing

In [ ]:
# 2. Define a narrow window to simulate holding W constant
tolerance = 0.05
w_constant_values = [-1, 0, 1]

for w_constant_val in w_constant_values:
    # Bin W values +- 0.5 (random estimate) of the constant vals, since there is only 1 exact value for W equal to a constant
    mask = (W >= w_constant_val - tolerance) & (W <= w_constant_val + tolerance)
    # Grab the corresponding values of X/Z/Y where W is in the defined bin
    X_for_constant_w = X[mask]
    Z_for_constant_w = Z[mask]
    Y_for_constant_w = Y[mask]
    print(f"For w value {w_constant_val} +-5 we found {len(X_for_constant_w)} rows")
    
    X_Z = np.column_stack((X_for_constant_w, Z_for_constant_w))
    X_Z = sm.add_constant(X_Z)
    model = sm.OLS(Y_for_constant_w, X_Z).fit()
    
    x_coef = model.params[1] 
    sample_count = np.sum(mask)
    print(f"{x_coef=}")
    print("\n")

For w value -1 +-5 we found 24 rows
x_coef=np.float64(1.5190315683721274)


For w value 0 +-5 we found 35 rows
x_coef=np.float64(0.6748644062383412)


For w value 1 +-5 we found 33 rows
x_coef=np.float64(1.0511341649016852)




A3. Option B. I don't see a consistent pattern

### Problem 4

Q4. 

```
def make_error(corr_const, num): 
          err = list() 
          prev = np.random.normal(0, 1) 
          for n in range(num): 
                   prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, 1) 
                   err.append(prev) 
          return np.array(err) 
```

Create a linear regression model that uses this function as the error for both (a) the treatment, ﻿X﻿, and (b) the outcome, ﻿Y﻿. (You can use random normal error for any other covariates, if you have them.) 

As corr_const increases from 0.2 to 0.5 to 0.8, find (i) the standard deviation of the estimate of the ﻿X﻿ coefficient over many trials, and (ii) the mean of the standard error estimate of the ﻿X﻿ coefficient over many trials. 

When corr_const increases, then: 

Hint: don't forget to include an intercept in your regression

Option A: (i) and (ii) differ, but their ratio remains about the same

Option B: The ratio (i) / (ii) decreases

Option C: The ratio (i) / (ii) increases

Option D: (i) and (ii) remain about the same

In [72]:
def make_error(corr_const, num): 
          err = list() 
          prev = np.random.normal(0, 1) 
          for n in range(num): 
                   prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, 1) 
                   err.append(prev) 
          return np.array(err) 

In [ ]:
for corr_const in [0.2, 0.5, 0.8]:
    print(corr_const)
    x_coef_estimates = []
    standard_error_estimates = []

    trial_count = 100
    for _ in range(trial_count):
        X = make_error(corr_const, trial_count)
        Y = X + make_error(corr_const, trial_count)
        X = sm.add_constant(X)

        model = sm.OLS(Y, sm.add_constant(X)).fit()
        x_coef_estimates.append(model.params[1])
        standard_error_estimates.append(model.bse[1])
        
    x_coef_estimate_std = np.std(x_coef_estimates)
    standard_error_estimates_mean = np.mean(standard_error_estimates)
    print(f"{x_coef_estimate_std=} {standard_error_estimates_mean=}")
    print('\n')

0.2
x_coef_estimate_std=np.float64(0.09550081473328027) standard_error_estimates_mean=np.float64(0.1026980589482819)


0.5
x_coef_estimate_std=np.float64(0.1346594099523361) standard_error_estimates_mean=np.float64(0.10158048776493164)


0.8
x_coef_estimate_std=np.float64(0.22610360240638133) standard_error_estimates_mean=np.float64(0.10151907282724473)




A4. Option C

### Reflection 7 Problem 1

Create a linear regression model involving a confounder that is left out of the model.  Show whether the true correlation between $$X$$ and $$Y$$ is overestimated, underestimated, or neither.  Explain in words why this is the case for the given coefficients you have chosen.

In [61]:
num = 10_000

# Simulate the lightning storm which has the first effects caused by it. We could model it as a 
# float (0 == no storm and from there it is a severity of a storm). 
# This seemed complicated so I make it binary instead (Yes or No storm) with equal prob
lightning_storm = np.random.binomial(1, 0.5, num)

# Lightning storms frighten bears, decreasing their population
# Create baseline population noise
bears = np.random.normal(1_000, 50, num) # Normal distribution centered around 1,000 with std 50
# Add effect of lighting storm = 1
bears = np.where(lightning_storm == 1, bears * 0.9, bears) # When lightning storm happens it decreases bears population by 10%

# Lightning storm frighten deer, decreasing their population
deer = np.random.normal(1_000, 50, num) # Normal distribution centered around 1,000 with std 50
deer = np.where(lightning_storm == 1, deer * 0.9, deer) # When lightning storm happens it decreases deer population by 10%

# Lightning storm cause flowers to grow, increasing their population.
flowers = np.random.normal(1_000, 50, num) # Normal distribution centered around 1,000 with std 50
flowers = np.where(lightning_storm == 1, flowers + 100, flowers) # When lightning storm happens it decreases deer by 100 (static value)

# Bears eat deer, decreasing their population.
deer = deer - (bears * 0.1) # Decrease deer count by 10% of the bear population. (random number I decided on shows effect)

# Deer eat flowers, decreasing their population.
flowers = flowers - (deer * 0.1) # Decrease flowers count by 10% of the deer population. (random number I decided on)

df = pd.DataFrame(
    {
        'lightning_storm': lightning_storm,
        'bears': bears,
        'deer': deer,
        'flowers': flowers
    }
)
df

,lightning_storm,bears,deer,flowers
0,0,991.509917,1010.975636,873.834298
1,1,910.897933,705.118336,1010.303358
2,0,1100.409477,925.690029,937.892061
3,1,953.072110,741.928628,966.438485
4,1,907.682698,787.513303,1003.405201
...,...,...,...,...
9995,1,953.560997,732.424426,1038.974798
9996,0,1020.590278,973.706905,846.367586
9997,1,900.065703,846.527613,911.946542
9998,0,1025.885357,926.409906,907.893165


In [62]:
# The confounder is lightning_storm, the X is bears, and the Y is deer

X_without_lightning= sm.add_constant(df['bears'])
model_without_lightning = sm.OLS(df['deer'], X_without_lightning).fit()

X_with_lightning = sm.add_constant(df[['bears', 'lightning_storm']])
model_with_lightning = sm.OLS(df['deer'], X_with_lightning).fit()

model_without_lightning.params['bears'], model_with_lightning.params['bears']

(np.float64(0.4286541375737182), np.float64(-0.0925454891367076))

We know that the actual coef is -0.10 (from the *0.1 calculation in sim code)

Without the confounder we estimate 0.41157357014404083, overestimating

With the confounder we estimate 0.09848556025687241 which is close to the true value